# 16 - Classical oriented-Gabor validation baseline

**Run All on Colab. CPU is sufficient; a T4 runtime is also fine. No training.**
After a runtime reset, restore validation images with notebook 02 and NBIS with notebook 04 on this same runtime. Completed notebook-14 and its recorded notebook-08/11 reference reports must remain on Drive. Do not rerun training.

Evaluates the pinned `fingerprint-enhancer 0.0.14` implementation of orientation/frequency-based Gabor enhancement, related to Hong, Wan and Jain, TPAMI 1998. This is a fixed public implementation baseline, not a claim of exact reproduction of the original paper. Source: https://github.com/Utkarsh-Deshmukh/Fingerprint-Enhancement-Python . The verified PyPI source archive and its BSD-2-Clause license are preserved unmodified on Drive. Only public code is downloaded; no fingerprint images are uploaded.

- Use the exact same normalized 500-dpi full images, synthetic cases, probe-to-gallery direction, frozen pairs and unchanged original galleries.
- No internal resizing. Fixed upstream defaults, `invert_output=True`: negative filter responses mark dark ridges; output is **binary black ridges on white**, not grayscale restoration. Polarity is fixed before scoring, never selected by recognition results.
- Flat/too-small/no-foreground/no-valid-frequency images are explicitly logged as Gabor failures. Their comparisons are rejected with missing scores, not zero scores, original-image fallbacks or fabricated templates. Unexpected exceptions and NBIS/infrastructure failures abort.
- Reuse all 40 verified notebook-14 variants, generate only 10 Gabor variants. CPU versus GPU is irrelevant for reuse here: no neural inference is rerun. NBIS binary hashes and normalization/simulator library versions must match; otherwise stop rather than mix builds.
- Primary: original-input thresholds fixed per database. Secondary: separate empirical conditional-FMR calibration is diagnostic, not equal achieved FAR/FMR. Include byte-distinct sensitivity, failures and original-image harms.
- Binary-output pixel MAE is only a representation diagnostic; it is not a fair grayscale restoration quality ranking. Gabor-specific failures are reported separately from insufficient minutiae.
- Completed condition variants resume from verified Drive files, even after local-cache loss. An unfinished condition is recomputed; same-runtime template caches avoid repeated extraction, but Gabor filtering itself can repeat. Do not interrupt a busy kernel or launch duplicate Run All.

This is exploratory validation on one frozen simulator, not evidence about real dry/wet labels, unseen degradations, held-out tests or final journal novelty. No threshold/loss tuning is performed here. Outputs: `evaluations/gabor_validation_v1/<hash>/`. Completion: `GABOR VALIDATION MATCHING COMPLETE`.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import re
import shutil
import subprocess
import sys
import types
import zipfile
import io
import tarfile
import urllib.request
import warnings
import time
import scipy
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive
DEVICE = torch.device('cpu')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
REFERENCE_DIR = PROJECT_ROOT / 'evaluations' / 'm1_controlled_validation_v1' / 'bef0c1363b1e6a721afc'
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT, BOZORTH3 = (NBIS_ROOT / 'install' / 'bin' / name for name in ('mindtct', 'bozorth3'))
METHODS = ('input', 'm0', 'l1_control', 'l1_sobel', 'gabor')
PRIOR_DIR = PROJECT_ROOT / 'evaluations' / 'm2_structure_validation_v1' / '1374b21b3b10b3202a8d'
CONDITIONS = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
COMPARISONS = [('input', 'gabor'), ('l1_control', 'gabor'), ('l1_sobel', 'gabor')]

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact checksum mismatch: {directory / name}')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
print('CPU Gabor baseline; no neural inference or training.')


In [ ]:
GABOR_URL = 'https://files.pythonhosted.org/packages/fb/38/2394cad384cb1837f1cbae86964c2fc3918f34e2e5fa3d39ffea2c75b858/fingerprint_enhancer-0.0.14.tar.gz'
GABOR_ARCHIVE_SHA = 'd033eaafd3332e783539e50b2643ee29da065cc2025a26a85d41e04239847840'
VENDOR_DIR = PROJECT_ROOT / 'tools' / 'fingerprint_enhancer_0_0_14'
VENDOR_DIR.mkdir(parents=True, exist_ok=True)
archive_path = VENDOR_DIR / 'fingerprint_enhancer-0.0.14.tar.gz'
if not archive_path.exists():
    with urllib.request.urlopen(GABOR_URL, timeout=120) as response:
        archive_data = response.read()
    if hashlib.sha256(archive_data).hexdigest() != GABOR_ARCHIVE_SHA:
        raise RuntimeError('Downloaded Gabor archive checksum mismatch.')
    write_verified(archive_path, archive_data)
if file_digest(archive_path) != GABOR_ARCHIVE_SHA:
    raise RuntimeError('Stored Gabor archive checksum mismatch.')
with tarfile.open(fileobj=io.BytesIO(archive_path.read_bytes()), mode='r:gz') as archive:
    for member, filename in (
        ('fingerprint_enhancer-0.0.14/src/fingerprint_enhancer/fingerprint_image_enhancer.py', 'fingerprint_image_enhancer.py'),
        ('fingerprint_enhancer-0.0.14/LICENSE', 'LICENSE')):
        info = archive.getmember(member)
        if not info.isfile() or info.size > 100000:
            raise RuntimeError('Unexpected Gabor archive member.')
        with archive.extractfile(info) as stream:
            data = stream.read()
        destination = VENDOR_DIR / filename
        if destination.exists() and destination.read_bytes() != data:
            raise RuntimeError(f'Pinned upstream file changed: {destination}')
        if not destination.exists():
            write_verified(destination, data)
gabor_module = load_module('pinned_fingerprint_gabor_0_0_14', VENDOR_DIR / 'fingerprint_image_enhancer.py')
gabor_parameters = {name: parameter.default for name, parameter in
    inspect.signature(gabor_module.FingerprintImageEnhancer).parameters.items()}
print('Verified upstream archive, unmodified module and BSD-2-Clause license.')
print('Fixed parameters:', gabor_parameters, '| resize=False | invert_output=True')


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
validation = locked[locked['split'].eq('validation')].sort_values('relative_path').reset_index(drop=True)
if validation.empty or set(validation['dataset']) != {'FVC2000', 'FVC2002'} or validation.groupby(['dataset', 'db']).ngroups != 6:
    raise RuntimeError('Expected complete source-only validation in six databases.')
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
prior_config = json.loads((PRIOR_DIR / 'configuration.json').read_text(encoding='utf-8'))
prior_completion = json.loads((PRIOR_DIR / 'completion.json').read_text(encoding='utf-8'))
prior_sha = file_digest(PRIOR_DIR / 'configuration.json')
if (prior_completion['status'] != 'structure_validation_matching_complete'
        or prior_completion['configuration_sha256'] != prior_sha or prior_completion['variants_available'] != 40
        or set(prior_completion['methods']) != set(METHODS) - {'gabor'}):
    raise RuntimeError('Expected completed notebook-14 evaluation.')
verify_files(PRIOR_DIR, prior_completion['file_sha256'])
if (prior_config['pair_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or prior_config['pair_csv_sha256'] != file_digest(PAIR_DIR / 'pairs.csv')
        or pair_config['pair_csv_sha256'] != file_digest(PAIR_DIR / 'pairs.csv')
        or prior_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (pair_config, sim_config))):
    raise RuntimeError('Frozen split/pairs/simulator provenance differs.')
helper_path = PRIOR_DIR / 'frozen_matching_helpers.py.txt'
if file_digest(helper_path) != prior_config['frozen_helpers_sha256']:
    raise RuntimeError('Frozen matching helper checksum mismatch.')
HELPER_SOURCE = helper_path.read_bytes().decode('utf-8')
helper_code = compile(HELPER_SOURCE, str(helper_path), 'exec')
reference_config, reference_completion = None, None
if prior_config['reference_reuse']:
    if (file_digest(REFERENCE_DIR / 'configuration.json') != prior_config['reference_evaluation_sha256']
            or file_digest(REFERENCE_DIR / 'completion.json') != prior_config['reference_completion_sha256']):
        raise RuntimeError('Notebook-11 nested reference changed.')
    reference_config = json.loads((REFERENCE_DIR / 'configuration.json').read_text(encoding='utf-8'))
    reference_completion = json.loads((REFERENCE_DIR / 'completion.json').read_text(encoding='utf-8'))
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
if not (NBIS_ROOT / 'verified_build.json').is_file():
    raise FileNotFoundError('NBIS runtime installation missing. Run notebook 04 first on this runtime.')
build = json.loads((NBIS_ROOT / 'verified_build.json').read_text(encoding='utf-8'))
if (build['signature']['source_sha256'] != pair_config['nbis_source_sha256']
        or pair_config['nbis_source_sha256'] != '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
        or pair_config['mindtct_flags'] != [] or pair_config['bozorth3_flags'] != ['-A', 'minminutiae=10']):
    raise RuntimeError('NBIS source/settings differ from the frozen pair protocol.')
for binary in (MINDTCT, BOZORTH3):
    if not binary.is_file() or file_digest(binary) != build['binary_sha256'][binary.name]:
        raise RuntimeError(f'NBIS binary missing/modified: {binary}')
    subprocess.run([str(binary), '-version'], capture_output=True, text=True, check=True)
pairs = pd.read_csv(PAIR_DIR / 'pairs.csv', keep_default_na=False)
if pairs['pair_id'].duplicated().any() or validation['relative_path'].duplicated().any() or pairs['duplicate_sensitivity_eligible'].dtype != bool:
    raise RuntimeError('Invalid pair/image keys or duplicate-sensitivity encoding.')
probe_records = validation[validation['impression_id'].ne(1)].to_dict('records')
gallery_records = validation[validation['impression_id'].eq(1)].to_dict('records')
if (set(pairs['probe_path']) != {row['relative_path'] for row in probe_records}
        or set(pairs['gallery_path']) != {row['relative_path'] for row in gallery_records}):
    raise RuntimeError('Frozen pairs do not cover all validation probes/galleries.')
for record in validation.to_dict('records'):
    path = DATA_ROOT / record['relative_path']
    if not path.is_file() or file_digest(path) != record['sha256']:
        raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02 on this runtime.')
model_path = M0_DIR / 'restoration_baseline.py'
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
if file_digest(model_path) != m0_config['source_file_sha256']['restoration_baseline.py']:
    raise RuntimeError('Frozen normalization helper module changed.')
if file_digest(M0_DIR / 'configuration.json') != prior_config['m0_configuration_sha256']:
    raise RuntimeError('Normalization module provenance differs from notebook14.')
training = load_module('gabor_normalization_v1', model_path)
simulator = load_module('gabor_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
if build['binary_sha256'] != prior_config['inference_and_matcher']['nbis_binary_sha256']:
    raise RuntimeError('NBIS build differs from notebook14. Do not mix builds; restore its build or rerun notebook14 on this build first.')
if (Image.__version__ != prior_config['inference_and_matcher']['pillow_version']
        or np.__version__ != prior_config['inference_and_matcher']['numpy_version']):
    raise RuntimeError('Normalization/simulator versions differ from notebook14.')
expected_prior_units = {f'{family}_{severity}_{method}' for family, severity in CONDITIONS for method in METHODS if method != 'gabor'}
if set(prior_completion['unit_provenance']) != expected_prior_units:
    raise RuntimeError('Notebook14 does not cover every required reference variant.')
print('Locked validation probes:', len(probe_records), '| unchanged original galleries:', len(gallery_records))
print('Reusing 40 verified prior variants; evaluating only ten Gabor variants.')


In [ ]:
def make_helpers(configuration_sha):
    module = types.ModuleType('frozen_structure_matching_helpers')
    names = ('torch', 'np', 'pd', 'Image', 'Path', 'hashlib', 'json', 'math', 're', 'shutil', 'subprocess', 'zipfile',
        'file_digest', 'json_bytes', 'write_verified', 'training', 'simulator', 'DATA_ROOT', 'pair_config',
        'sim_config', 'pairs', 'probe_records', 'MINDTCT', 'BOZORTH3', 'DEVICE')
    module.__dict__.update({name: globals()[name] for name in names})
    module.evaluation_sha, module.TEMPLATE_DIR, module.model = configuration_sha, TEMPLATE_DIR, None
    exec(helper_code, module.__dict__)
    return module

def get_unit(family, severity, method):
    if method not in METHODS:
        raise ValueError(f'Unknown method: {method}')
    unit = f'{family}_{severity}_{method}'
    if unit in UNIT_CACHE:
        return UNIT_CACHE[unit]
    reused = method != 'gabor'
    if reused:
        provenance = prior_completion['unit_provenance'][unit]
        directory = Path(provenance['directory'])
        if file_digest(directory / 'completion.json') != provenance['completion_sha256']:
            raise RuntimeError(f'Prior variant marker changed: {unit}')
        if provenance['origin'] == 'notebook14':
            expected_sha = prior_sha
        elif provenance['origin'] == 'prior_verified_reference':
            if reference_completion is None or method not in ('input', 'm0'):
                raise RuntimeError(f'Unexpected nested reference: {unit}')
            original = reference_completion['unit_provenance'][unit]
            if (Path(original['directory']) != directory or original['completion_sha256'] != provenance['completion_sha256']
                    or original['condition'] != provenance['condition'] or original['method'] != method):
                raise RuntimeError(f'Nested reference provenance changed: {unit}')
            if original['origin'] == 'notebook08':
                expected_sha = reference_config['reference_evaluation_sha256']
            elif original['origin'] == 'notebook11':
                expected_sha = prior_config['reference_evaluation_sha256']
            else:
                raise RuntimeError(f'Unknown nested reference origin: {unit}')
        else:
            raise RuntimeError(f'Unknown prior variant origin: {unit}')
        if expected_sha not in reference_helpers:
            reference_helpers[expected_sha] = make_helpers(expected_sha)
        value = reference_helpers[expected_sha].load_unit(directory)
        if value is None:
            raise RuntimeError(f'Completed prior variant missing: {unit}')
    else:
        directory = REPORT_DIR / unit
        value = active_helpers.load_unit(directory)
    if value is not None:
        UNIT_CACHE[unit] = value
        UNIT_PROVENANCE[unit] = {'condition': f'{family}/{severity}', 'method': method,
            'origin': 'notebook14_verified_reference' if reused else 'notebook16', 'directory': str(directory),
            'completion_sha256': file_digest(directory / 'completion.json')}
    return value

def paired_rows(before, after, thresholds, condition, baseline_method, method):
    rows = active_helpers.comparison_rows(before, after, thresholds, condition)
    for row in rows:
        row['baseline_method'], row['method'] = baseline_method, method
        row['baseline_attempt_frr'] = row.pop('input_attempt_frr')
        row['method_attempt_frr'] = row.pop('m0_attempt_frr')
    return rows
def gabor_restore(image):
    if (image.ndim != 2 or not np.isfinite(image).all() or image.size == 0
            or image.min() < 0 or image.max() > 1):
        raise ValueError('Expected a finite nonempty 2D grayscale image in [0,1].')
    quantized = active_helpers.quantize(image)
    if min(image.shape) <= gabor_parameters['ridge_freq_blksze']:
        return None, 'gabor_image_too_small', ''
    if quantized.std() == 0:
        return None, 'gabor_flat_input', ''
    enhancer = gabor_module.FingerprintImageEnhancer(**gabor_parameters)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        enhancer._FingerprintImageEnhancer__ridge_segment(quantized)
        if not enhancer._mask.any() or not np.isfinite(enhancer._normim).all():
            return None, 'gabor_no_valid_foreground', '; '.join(str(item.message) for item in caught)
        enhancer._FingerprintImageEnhancer__ridge_orient()
        enhancer._FingerprintImageEnhancer__ridge_freq()
        if (not np.isfinite(enhancer._freq).all() or not (enhancer._freq > 0).any()
                or not np.isfinite(enhancer._mean_freq) or enhancer._mean_freq <= 0):
            return None, 'gabor_no_valid_frequency', '; '.join(str(item.message) for item in caught)
        enhancer._FingerprintImageEnhancer__ridge_filter()
    output = np.logical_not(enhancer._binim).astype(np.float32)
    if output.shape != image.shape or not np.isfinite(output).all():
        raise RuntimeError('Unexpected upstream Gabor output shape/finiteness.')
    return output, 'ok', '; '.join(str(item.message) for item in caught)

def gabor_row(record, family, severity):
    reference, degraded = active_helpers.condition_input(record, family, severity)
    started = time.perf_counter()
    output, processing_status, warning_text = gabor_restore(degraded)
    elapsed = time.perf_counter() - started
    if output is None:
        template = {'status': processing_status, 'minutiae': 0, 'xyt_path': '',
                    'png_sha256': '', 'template_sha256': '', 'width': degraded.shape[1], 'height': degraded.shape[0]}
        values = {'float_input_mae': float(np.abs(degraded - reference).mean()),
            'png_input_mae': float(np.abs(active_helpers.quantize(degraded).astype(float) -
                active_helpers.quantize(reference)).mean() / 255),
            **{name: np.nan for name in ('float_output_mae', 'float_change_mae', 'png_output_mae', 'changed_pixel_fraction')},
            'png_identical_to_input': False}
    else:
        template = active_helpers.extract_png(active_helpers.quantize(output))
        values = active_helpers.pixel_metrics(reference, degraded, output)
    return {**record, **template, **values, 'condition': f'{family}/{severity}', 'method': 'gabor',
        'gabor_processing_status': processing_status, 'gabor_warnings': warning_text, 'gabor_seconds': elapsed}


In [ ]:
adapter_source = '\n\n'.join(inspect.getsource(function) for function in (make_helpers, get_unit, paired_rows, gabor_restore, gabor_row))
evaluation_config = {'version': 'gabor_validation_v1', 'reference_evaluation_sha256': prior_sha,
    'reference_completion_sha256': file_digest(PRIOR_DIR / 'completion.json'),
    'pair_configuration_sha256': file_digest(PAIR_DIR / 'configuration.json'), 'pair_csv_sha256': file_digest(PAIR_DIR / 'pairs.csv'),
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'frozen_helpers_sha256': file_digest(helper_path), 'adapters_sha256': hashlib.sha256(adapter_source.encode()).hexdigest(),
    'gabor': {'archive_sha256': GABOR_ARCHIVE_SHA, 'source_sha256': file_digest(VENDOR_DIR / 'fingerprint_image_enhancer.py'),
        'license_sha256': file_digest(VENDOR_DIR / 'LICENSE'), 'parameters': gabor_parameters,
        'resize': False, 'invert_output': True, 'output': 'binary black ridges on white; no grayscale reconstruction',
        'invalid_cases': 'explicit rejected attempts; no original-image fallback; unexpected exceptions abort'},
    'libraries': {'numpy': np.__version__, 'pillow': Image.__version__, 'scipy': scipy.__version__,
        'opencv': cv2.__version__, 'torch': str(torch.__version__), 'python': sys.version},
    'nbis_binary_sha256': build['binary_sha256'], 'methods': METHODS, 'conditions': CONDITIONS,
    'comparisons': COMPARISONS, 'training_performed': False, 'test_evaluated': False}
evaluation_data = json_bytes(evaluation_config)
evaluation_sha = hashlib.sha256(evaluation_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'evaluations' / 'gabor_validation_v1' / evaluation_sha[:20]
CACHE_DIR = LOCAL_ROOT / 'gabor_validation_v1' / evaluation_sha[:20]
TEMPLATE_DIR = CACHE_DIR / 'templates'
for directory in (REPORT_DIR, CACHE_DIR, TEMPLATE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
frozen_files = {'configuration.json': evaluation_data, 'frozen_matching_helpers.py.txt': HELPER_SOURCE.encode(),
    'evaluation_adapters.py.txt': adapter_source.encode()}
for name, data in frozen_files.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen Gabor evaluation changed: {path}')
    if not path.exists():
        write_verified(path, data)
active_helpers = make_helpers(evaluation_sha)
reference_helpers, UNIT_CACHE, UNIT_PROVENANCE = {}, {}, {}
print('Frozen Gabor evaluation:', evaluation_sha)
print('Reports/resume directory:', REPORT_DIR)


In [ ]:
pending = [(family, severity, method) for family, severity in CONDITIONS for method in METHODS
           if get_unit(family, severity, method) is None]
galleries = None
if pending:
    gallery_rows = []
    for record in gallery_records:
        reference, _ = active_helpers.condition_input(record, 'identity', 'none')
        gallery_rows.append({**record, **active_helpers.extract_png(active_helpers.quantize(reference))})
    galleries = pd.DataFrame(gallery_rows)
    write_csv(REPORT_DIR / 'gallery_extraction.csv', galleries)
    if True:
        columns = ['relative_path', 'png_sha256', 'template_sha256', 'minutiae', 'status']
        previous = pd.read_csv(PRIOR_DIR / 'gallery_extraction.csv')[columns].sort_values('relative_path').reset_index(drop=True)
        current = galleries[columns].sort_values('relative_path').reset_index(drop=True)
        if not current.equals(previous):
            raise RuntimeError('Original gallery extraction differs; do not mix reference scores.')
    print('Fixed original galleries prepared:', len(galleries), '| Pending variants:', len(pending), flush=True)
for family, severity in CONDITIONS:
    for method in METHODS:
        unit = f'{family}_{severity}_{method}'
        if get_unit(family, severity, method) is not None:
            print('Reused/resumed verified variant:', unit, flush=True)
            continue
        directory, work = REPORT_DIR / unit, CACHE_DIR / unit
        directory.mkdir(parents=True, exist_ok=True)
        work.mkdir(parents=True, exist_ok=True)
        rows = []
        for index, record in enumerate(probe_records, start=1):
            row = gabor_row(record, family, severity)
            rows.append(row)
            if row['gabor_processing_status'] != 'ok':
                print('GABOR FAILURE:', unit, record['relative_path'], row['gabor_processing_status'], flush=True)
            if row['gabor_warnings']:
                print('GABOR WARNING:', unit, record['relative_path'], row['gabor_warnings'], flush=True)
            if index % 100 == 0 or index == len(probe_records):
                write_csv(directory / 'extraction_progress.csv', pd.DataFrame(rows))
                print(f'{unit}: extracted/cached {index}/{len(probe_records)} probes', flush=True)
        extraction = pd.DataFrame(rows)
        scored = active_helpers.match_pairs(extraction, galleries, work)
        write_csv(directory / 'scores.csv', scored)
        write_csv(directory / 'extraction.csv', extraction)
        active_helpers.archive_templates(directory / 'templates_and_logs.zip', [row for row in rows if row['xyt_path']] + galleries.to_dict('records'), work)
        marker = {'configuration_sha256': evaluation_sha, 'variant': unit,
            'file_sha256': {name: file_digest(directory / name) for name in ('scores.csv', 'extraction.csv', 'templates_and_logs.zip')}}
        write_verified(directory / 'completion.json', json_bytes(marker))
        get_unit(family, severity, method)
        print('Saved variant:', unit, '| rejected comparisons:', int((~scored['matchable']).sum()), flush=True)
write_csv(REPORT_DIR / 'unit_provenance.csv', pd.DataFrame(UNIT_PROVENANCE.values()))
print('All 50 method/condition variants verified; no neural inference, training or gallery enhancement.')

In [ ]:
original, _ = get_unit('identity', 'none', 'input')
fixed_thresholds = {key: active_helpers.calibrate_threshold(group.loc[group['matchable'] & group['label'].eq(0), 'score'])
                    for key, group in original.groupby(['dataset', 'db'], sort=True)}
metric_rows, pixel_rows, paired_changes, threshold_rows, curves = [], [], [], [], []
for family, severity in CONDITIONS:
    condition = f'{family}/{severity}'
    scores = {}
    for method in METHODS:
        scored, extraction = get_unit(family, severity, method)
        scores[method] = scored
        for (dataset, db), images in extraction.groupby(['dataset', 'db'], sort=True):
            pixel_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'probe_images': len(images), 'insufficient_minutiae': int(images['status'].eq('insufficient_minutiae').sum()),
                'gabor_processing_failures': int(images['status'].str.startswith('gabor_').sum()),
                'image_failure_rate': float(images['status'].ne('ok').mean()),
                **{name: float(images[name].mean()) for name in ('float_input_mae', 'float_output_mae', 'float_change_mae',
                    'png_input_mae', 'png_output_mae', 'changed_pixel_fraction')},
                'png_identical_image_fraction': float(images['png_identical_to_input'].mean())})
        for (dataset, db), group in scored.groupby(['dataset', 'db'], sort=True):
            valid_impostors = group.loc[group['matchable'] & group['label'].eq(0), 'score']
            calibrated = active_helpers.calibrate_threshold(valid_impostors) if len(valid_impostors) else None
            available = group.loc[group['matchable'], 'label'].nunique() == 2
            eer, gap = None, None
            if available:
                curve = active_helpers.conditional_curve(group)
                point = curve.loc[(curve['conditional_fmr'] - curve['conditional_fnmr']).abs().idxmin()]
                eer = float((point['conditional_fmr'] + point['conditional_fnmr']) / 2)
                gap = float(abs(point['conditional_fmr'] - point['conditional_fnmr']))
                curves.append(curve.assign(condition=condition, method=method, dataset=dataset, db=db))
            if calibrated is None or not available:
                print(f'WARNING: conditional calibration/EER unavailable for {condition}/{method}/{dataset}/{db}; all attempts retained.')
            threshold_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'original_fixed_threshold': fixed_thresholds[(dataset, db)], 'condition_calibrated_threshold': calibrated,
                'calibration_status': 'ok' if calibrated is not None else 'no_matchable_impostors'})
            for policy, threshold in (('original_fixed', fixed_thresholds[(dataset, db)]), ('condition_calibrated', calibrated)):
                if threshold is None:
                    continue
                for view, selected in (('all_impressions', group),
                                       ('byte_distinct_sensitivity', group[group['duplicate_sensitivity_eligible']])):
                    values = active_helpers.metrics_at_threshold(selected, threshold)
                    if policy == 'condition_calibrated' and view == 'all_impressions' and values['conditional_fmr'] > 0.01 + 1e-12:
                        raise RuntimeError('Condition-calibrated validation FMR exceeded 1%.')
                    metric_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                        'threshold_policy': policy, 'view': view, **values,
                        'conditional_eer_discrete_approx': eer if view == 'all_impressions' else None,
                        'eer_crossing_gap': gap if view == 'all_impressions' else None})
    for baseline_method, method in COMPARISONS:
        paired_changes.extend(paired_rows(scores[baseline_method], scores[method], fixed_thresholds, condition, baseline_method, method))
metrics, pixels, paired = pd.DataFrame(metric_rows), pd.DataFrame(pixel_rows), pd.DataFrame(paired_changes)
write_csv(REPORT_DIR / 'validation_metrics.csv', metrics)
write_csv(REPORT_DIR / 'pixel_and_failure_metrics.csv', pixels)
write_csv(REPORT_DIR / 'paired_genuine_changes.csv', paired)
write_csv(REPORT_DIR / 'validation_thresholds.csv', pd.DataFrame(threshold_rows))
write_csv(REPORT_DIR / 'conditional_curves.csv', pd.concat(curves, ignore_index=True))
macro = metrics[metrics['view'].eq('all_impressions')].groupby(['condition', 'method', 'threshold_policy'], sort=True).agg(
    databases=('db', 'size'), macro_attempt_frr=('attempt_frr', 'mean'), macro_attempt_far=('attempt_far', 'mean'),
    macro_conditional_fnmr=('conditional_fnmr', 'mean'), valid_fnmr_databases=('conditional_fnmr', 'count'),
    macro_conditional_fmr=('conditional_fmr', 'mean'), valid_fmr_databases=('conditional_fmr', 'count')).reset_index()
if not macro['databases'].eq(6).all():
    raise RuntimeError('Macro summaries must include all six databases.')
write_csv(REPORT_DIR / 'macro_operating_points.csv', macro)
write_csv(REPORT_DIR / 'macro_fixed_threshold_metrics.csv', macro[macro['threshold_policy'].eq('original_fixed')])
aggregate_rows = []
scopes = {'synthetic_nine': [f'{family}/{severity}' for family, severity in CONDITIONS if family != 'identity'],
    **{family: [f'{name}/{severity}' for name, severity in CONDITIONS if name == family] for family in ('dry_like', 'wet_like', 'generic')}}
for scope, members in scopes.items():
    frame = macro[macro['condition'].isin(members)]
    for (method, policy), group in frame.groupby(['method', 'threshold_policy'], sort=True):
        if len(group) != len(members):
            raise RuntimeError('Incomplete aggregate condition coverage.')
        aggregate_rows.append({'scope': scope, 'method': method, 'threshold_policy': policy,
            'conditions': len(group), **{name: float(group[name].mean()) for name in
                ('macro_attempt_frr', 'macro_attempt_far', 'macro_conditional_fnmr', 'macro_conditional_fmr')},
            'complete_conditional_metrics': bool(group['valid_fnmr_databases'].eq(6).all() and group['valid_fmr_databases'].eq(6).all())})
aggregates = pd.DataFrame(aggregate_rows)
write_csv(REPORT_DIR / 'aggregate_operating_points.csv', aggregates)
print('Equal condition/database weights; condition-calibrated FMR <= 1% is NOT identical achieved FMR or independent test performance.')
print(aggregates.to_string(index=False))
print('\nOriginal-image harm/recovery at original fixed thresholds:')
print(paired[paired['condition'].eq('identity/none')].groupby(['baseline_method', 'method'])[['acceptance_recoveries', 'acceptance_harms']].sum().to_string())
print('\nGabor versus L1/Sobel at original fixed thresholds:')
print(paired[paired['baseline_method'].isin(['l1_control', 'l1_sobel'])].to_string(index=False))

In [ ]:
gabor_extraction = pd.concat([get_unit(family, severity, 'gabor')[1] for family, severity in CONDITIONS], ignore_index=True)
write_csv(REPORT_DIR / 'gabor_processing_diagnostics.csv', gabor_extraction[['relative_path', 'condition',
    'gabor_processing_status', 'gabor_warnings', 'gabor_seconds', 'status', 'minutiae']])
print('Processing status counts:')
print(gabor_extraction.groupby(['condition', 'gabor_processing_status']).size().to_string())
print('Binary-output MAE is not comparable as a grayscale-restoration quality ranking.')
preview = None
for family, severity in CONDITIONS:
    successful = get_unit(family, severity, 'gabor')[1]
    successful = successful[successful['gabor_processing_status'].eq('ok')]
    if len(successful):
        preview = (family, severity, successful.iloc[0].to_dict())
        break
if preview is not None:
    family, severity, record = preview
    reference, degraded = active_helpers.condition_input(record, family, severity)
    restored, status, warning_text = gabor_restore(degraded)
    if status != 'ok':
        raise RuntimeError('Previously successful Gabor preview no longer processes.')
    figure, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, title, image in zip(axes, ('Original reference', f'Input {family}/{severity}', 'Binary Gabor'), (reference, degraded, restored)):
        axis.imshow(image, cmap='gray', vmin=0, vmax=1)
        axis.set_title(title)
        axis.axis('off')
    figure.tight_layout()
    figure.savefig(REPORT_DIR / 'gabor_preview.png', dpi=140)
    plt.show()
    plt.close(figure)
else:
    print('WARNING: all Gabor probes failed; no preview or successful-case quality claim is possible.')
if len(UNIT_PROVENANCE) != 50:
    raise RuntimeError('Missing completed variants.')
files = [*frozen_files, 'unit_provenance.csv', 'gallery_extraction.csv', 'validation_metrics.csv',
    'pixel_and_failure_metrics.csv', 'paired_genuine_changes.csv', 'validation_thresholds.csv', 'conditional_curves.csv',
    'macro_operating_points.csv', 'macro_fixed_threshold_metrics.csv', 'aggregate_operating_points.csv',
    'gabor_processing_diagnostics.csv']
if preview is not None:
    files.append('gabor_preview.png')
completion = {'status': 'gabor_validation_matching_complete', 'configuration_sha256': evaluation_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in files}, 'unit_provenance': UNIT_PROVENANCE,
    'methods': METHODS, 'variants_available': 50, 'pairs_per_variant': len(pairs),
    'training_performed': False, 'test_evaluated': False, 'independent_test_improvement_verified': False,
    'interpretation': 'fixed implementation baseline; binary output; exploratory synthetic validation, no real moisture claims'}
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
print('GABOR VALIDATION MATCHING COMPLETE:', REPORT_DIR)
print('Send aggregate operating points, original-image harms and Gabor processing diagnostics. Keep negative outcomes.')
